# Load the data

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


file_path = "./Dataset/training/la_mmh_training_lms_item_activity_by_session_fact.csv"
df_lms = pd.read_csv(file_path)
print(df_lms.shape)

file_path_2 = "./Dataset/training/la_mmh_training_video_activity_by_view_fact.csv"
df_activity = pd.read_csv(file_path_2)
print(df_activity.shape)

file_path_3 = "./Dataset/training/la_mmh_training_person_dim.csv"
df_deregistrered = pd.read_csv(file_path_3)
print(df_deregistrered.shape)


file_path_4 = "./Dataset/training/la_mmh_training_person_module_instance_fact.csv"
df_module = pd.read_csv(file_path_4)
print(df_module.shape)

(7857424, 8)
(1145358, 7)
(14951, 3)
(100611, 4)


# Find the phantom

## LMS and activity

In [60]:
import pandas as pd

# 1. Trouver les lignes dupliquées
duplicated_rows_lms = df_lms[df_lms.duplicated(keep=False)]
duplicated_rows_activity = df_activity[df_activity.duplicated(keep=False)]

# 2. Extraire les P_ID uniques de chaque dataset
unique_pids_lms = set(duplicated_rows_lms['P_ID'].unique())
unique_pids_activity = set(duplicated_rows_activity['P_ID'].unique())

# 3. Combiner les deux ensembles (union) pour n'avoir qu'une seule liste unique de P_ID
all_unique_pids = sorted(list(unique_pids_lms.union(unique_pids_activity)))

# Afficher le résultat
print(f"Nombre total d'étudiants uniques combinés (LMS + Activity) : {len(all_unique_pids)}")

# 4. Sauvegarder dans un unique fichier CSV (un P_ID par ligne)
df_combined = pd.DataFrame({'P_ID': all_unique_pids})
df_combined.to_csv("all_duplicated_pids.csv", index=False)

print("Fichier 'all_duplicated_pids.csv' sauvegardé avec succès !")

Nombre total d'étudiants uniques combinés (LMS + Activity) : 77
Fichier 'all_duplicated_pids.csv' sauvegardé avec succès !


## Video activity

# Find the undead students

In [62]:
df_deregistered_only = df_deregistrered[
    df_deregistrered['deregistration_date'].notna() & (df_deregistrered['deregistration_date'] != 'null')
    ]

print(f"Total number of deregristered person : {len(df_deregistered_only)}")

#df_deregistered_only['deregistration_date'] = pd.to_datetime(df_deregistered_only['deregistration_date'])
#df_activity['DATE'] = pd.to_datetime(df_activity['DATE'])

merged_df = pd.merge(df_activity, df_deregistered_only, on='P_ID', how='inner')

violations = merged_df[merged_df['DATE'] > merged_df['deregistration_date']]


# 1. Compter le nombre d'activités (lignes)
print(f"Nombre total d'activités trouvées après désinscription : {len(violations)}")

# 2. Compter le nombre d'étudiants UNIQUES concernés
unique_violators = violations['P_ID'].unique()
print(f"Nombre d'étudiants uniques actifs après désinscription : {len(unique_violators)}")

# 3. Sauvegarder les P_IDs uniques, chacun sur une ligne séparée (utile pour soumettre)
pd.DataFrame({'P_ID': unique_violators}).to_csv("students_active_after_deregistration_pids.csv", index=False)

# Sauvegarder le détail complet des anomalies si tu veux garder les lignes
violations.to_csv("activites_post_desinscription_anomalies.csv", index=False)


Total number of deregristered person : 196
Nombre total d'activités trouvées après désinscription : 961
Nombre d'étudiants uniques actifs après désinscription : 79


# Find the doppelganger

In [53]:
import pandas as pd

# 1. Créer une signature unique pour chaque action (ex: combinaison de VIDEO_ID et DATE ou TIMESTAMP)
# Tu peux ajuster les colonnes selon ce qui définit l'activité exacte
df_activity['activity_signature'] = (
    df_activity['VIDEO_ID'].astype(str) + '_' + 
    df_activity['DATE'].astype(str) + '_' + 
    df_activity['TIMESTAMP'].astype(str)
)

# 2. Regrouper par P_ID pour obtenir la liste complète (et triée) des activités de chaque étudiant
student_profiles = (
    df_activity.groupby('P_ID')['activity_signature']
    .apply(lambda x: tuple(sorted(list(x))))
    .reset_index(name='profile')
)

# 3. Trouver les profils identiques partagés par plusieurs P_ID
doppelgangers = student_profiles[
    student_profiles.duplicated(subset=['profile'], keep=False)
].sort_values('profile')

# --- Compter chaque paire/groupe individuellement ---
grouped_doppelgangers = doppelgangers.groupby('profile')

print(f"📊 Nombre total de paires/groupes détectés : {len(grouped_doppelgangers)}")
print(f"👥 Nombre total d'étudiants concernés : {len(doppelgangers)}\n")

print("--- DÉTAIL DE CHAQUE PAIRE / GROUPE ---")
for group_id, (profile, group) in enumerate(grouped_doppelgangers, 1):
    pids = group['P_ID'].tolist()
    print(f"Paire #{group_id} ({len(pids)} étudiants) : {', '.join(pids)}")

# 4. Afficher les P_IDs sur une ligne séparée (format d'origine)
print("\n--- PAIRES DE DOPPELGÄNGERS (UN P_ID PAR LIGNE) ---")
for _, group in grouped_doppelgangers:
    for p_id in group['P_ID']:
        print(p_id)
    print("-" * 15)  # Séparateur visuel entre chaque paire

# Sauvegarder la liste des P_IDs uniques pour la soumission
doppelgangers[['P_ID']].to_csv("doppelgangers_pids.csv", index=False)

📊 Nombre total de paires/groupes détectés : 30
👥 Nombre total d'étudiants concernés : 60

--- DÉTAIL DE CHAQUE PAIRE / GROUPE ---
Paire #1 (2 étudiants) : P00010770, P00907330
Paire #2 (2 étudiants) : P00012287, P00365560
Paire #3 (2 étudiants) : P00003382, P00922502
Paire #4 (2 étudiants) : P00021322, P00948045
Paire #5 (2 étudiants) : P00336066, P00002959
Paire #6 (2 étudiants) : P00624727, P00019901
Paire #7 (2 étudiants) : P00007309, P00219495
Paire #8 (2 étudiants) : P00017482, P00944765
Paire #9 (2 étudiants) : P00408220, P00016841
Paire #10 (2 étudiants) : P00000512, P00116388
Paire #11 (2 étudiants) : P00004030, P00159448
Paire #12 (2 étudiants) : P00659653, P00010434
Paire #13 (2 étudiants) : P00336081, P00013039
Paire #14 (2 étudiants) : P00008029, P00955083
Paire #15 (2 étudiants) : P00959179, P00009037
Paire #16 (2 étudiants) : P00006439, P00209491
Paire #17 (2 étudiants) : P00901311, P00011616
Paire #18 (2 étudiants) : P00001387, P00647364
Paire #19 (2 étudiants) : P000114

# Find the SHAPE-SHIFTER

In [54]:
import pandas as pd

# 1. Charger les tables nécessaires (adapte les chemins selon ton arborescence)
# df_module_registration contient : PMI_ID, P_ID, MODULE_INSTANCE_ID, MODULE_REGISTRATION_STATUS
print(df_module)

# 2. Si le programme est codé dans le MODULE_INSTANCE_ID ou via une table de référence :
# Par exemple, extrayons un préfixe ou identifiant de programme du MODULE_INSTANCE_ID 
# (À adapter selon la structure de tes IDs de modules, ex: les premiers caractères)
df_module['programme_proxy'] = df_module['MODULE_INSTANCE_ID'].astype(str).str.extract(r'([A-Za-z]+)')

# 3. Regrouper par étudiant et lister les programmes/modules uniques auxquels ils sont inscrits
student_programmes = df_module.groupby('P_ID')['programme_proxy'].unique().reset_index()

# 4. Filtrer les étudiants qui ont plus d'un programme distinct
shape_shifters = student_programmes[student_programmes['programme_proxy'].apply(len) > 1]

print(f"🔍 Nombre d'étudiants ayant changé de programme : {len(shape_shifters)}")
print(shape_shifters.head())

# 5. Sauvegarder la liste des P_ID (un par ligne) pour soumission
shape_shifters[['P_ID']].to_csv("shape_shifters_pids.csv", index=False)
print("Fichier 'shape_shifters_pids.csv' généré avec succès !")

            PMI_ID       P_ID MODULE_INSTANCE_ID MODULE_REGISTRATION_STATUS
0       PMI0153397  P00012615           I2301485                  Mandatory
1       PMI0062532  P00006447           I2301485                  Mandatory
2       PMI0090410  P00016549           I2302486                  Mandatory
3       PMI0074021  P00014949           I2301509                   Elective
4       PMI0082006  P00011861           I2302059                  Mandatory
...            ...        ...                ...                        ...
100606  PMI0119444  P00009944           I2300367                  Mandatory
100607  PMI0101257  P00008381           I2300367                  Mandatory
100608  PMI0128687  P00004674           I2301091                   Elective
100609  PMI0155492  P00017112           I2300367                  Mandatory
100610  PMI0109871  P00000753           I2300367                  Mandatory

[100611 rows x 4 columns]
🔍 Nombre d'étudiants ayant changé de programme : 0
Empty Data

In [71]:
"""
THE SHAPE-SHIFTER — find students who appear to switch programme mid-year.

Key problem: neither of your activity files has an explicit "PROGRAMME_ID"
or "MODULE_INSTANCE_ID" — only item-level identifiers (LMS_ITEM_ID for
session activity, VIDEO_ID for video views). So we can't sort by date and
spot a programme change directly.

Trick used here: a "programme" is really just a cohort that shares a common
bundle of content items. So we:
  1. Build, for every student, the SET of item IDs they interacted with.
  2. Build a graph of items: two items are linked (weighted) by how many
     students interact with BOTH. Items belonging to the same programme get
     taken together a lot -> they end up densely connected.
  3. Run community detection on that graph -> each community ≈ one programme
     cohort's content bundle.
  4. For each student, count how many of these detected "programme clusters"
     their items fall into. A normal student is ~100% in one cluster.
     A student with a real second cluster of 2+ items (not just one stray
     extra) is the anomaly -> looks like they switched programme partway
     through the year.

Inputs expected:
  - df_activity       -> one of your activity fact tables (session or video).
                         It has NO MODULE_INSTANCE_ID column; the real item
                         column is auto-detected from ITEM_COL candidates
                         below (LMS_ITEM_ID or VIDEO_ID) and used as the
                         "module" proxy. If you have BOTH activity files,
                         load each separately and concatenate (see note
                         after loading) for a combined view.
  - df_deregistrered  -> optional, P_ID + registration/deregistration dates,
                         only used to annotate flagged students.

Output:
  - shape_shifters.csv  -> ranked list of suspect P_IDs with evidence
"""

import pandas as pd
import networkx as nx
import networkx.algorithms.community as nxcom
from itertools import combinations
from collections import defaultdict, Counter

# ---- 1. Load data -----------------------------------------------------
pmi = df_activity
pdim = df_deregistrered if 'df_deregistrered' in dir() else None

# If you want to combine BOTH activity files into one richer signal, do this
# before running the rest of the script instead of using a single df_activity:
#
#   sess = df_sessions.rename(columns={'LMS_ITEM_ID': 'ITEM_ID'})[['P_ID', 'ITEM_ID']]
#   vid  = df_videos.rename(columns={'VIDEO_ID': 'ITEM_ID'})[['P_ID', 'ITEM_ID']]
#   pmi  = pd.concat([sess, vid], ignore_index=True)
#   ITEM_COL = 'ITEM_ID'   # skip auto-detection below if you do this

# Auto-detect which column is the item/content proxy for "module".
# Add other candidate names here if your file uses something else.
CANDIDATE_ITEM_COLS = ['MODULE_INSTANCE_ID', 'LMS_ITEM_ID', 'VIDEO_ID', 'ITEM_ID']
ITEM_COL = next((c for c in CANDIDATE_ITEM_COLS if c in pmi.columns), None)
if ITEM_COL is None:
    raise KeyError(
        f"None of {CANDIDATE_ITEM_COLS} found in df_activity columns: "
        f"{list(pmi.columns)}. Update CANDIDATE_ITEM_COLS."
    )
print(f"Using '{ITEM_COL}' as the module/content proxy column.")

# ---- 2. Student -> set of items -----------------------------------------
student_modules = pmi.groupby('P_ID')[ITEM_COL].apply(set)

# SCALABILITY GUARDRAILS -----------------------------------------------
# At item-level granularity (LMS_ITEM_ID / VIDEO_ID) cardinality is huge:
#  - a student touching N items contributes N*(N-1)/2 pairs. A handful of
#    students with hundreds of items each can blow this up to 10s/100s of
#    millions of edges, and greedy_modularity_communities is ~O(n^2 log n)
#    on top of that -> effectively hangs forever.
# Mitigations applied below:
#  1. Drop items touched by very few students (noise) and items touched by
#     almost everyone (hubs -> no discriminating power, but create a huge
#     number of edges).
#  2. Only keep edges with co-occurrence weight >= MIN_EDGE_WEIGHT.
#  3. Use Louvain (near-linear) instead of greedy modularity.
MIN_STUDENTS_PER_ITEM = 5          # drop items seen by fewer students than this
MAX_ITEM_POPULARITY_FRAC = 0.5     # drop items seen by more than this fraction of students
MIN_EDGE_WEIGHT = 3                # keep item-pairs co-taken by at least this many students

n_students = student_modules.shape[0]
item_student_count = Counter()
for mods in student_modules:
    item_student_count.update(mods)

keep_items = {
    item for item, cnt in item_student_count.items()
    if cnt >= MIN_STUDENTS_PER_ITEM and cnt <= MAX_ITEM_POPULARITY_FRAC * n_students
}
print(f"Keeping {len(keep_items)} / {len(item_student_count)} items after pruning "
      f"(dropped rare items < {MIN_STUDENTS_PER_ITEM} students and hubs > "
      f"{MAX_ITEM_POPULARITY_FRAC:.0%} of students)")

student_modules_pruned = student_modules.apply(lambda mods: mods & keep_items)
student_modules_pruned = student_modules_pruned[student_modules_pruned.apply(len) > 0]

# ---- 3. Build item co-registration graph --------------------------------
edge_w = defaultdict(int)
for mods in student_modules_pruned:
    mods = sorted(mods)
    for a, b in combinations(mods, 2):
        edge_w[(a, b)] += 1

G = nx.Graph()
G.add_nodes_from(keep_items)
for (a, b), w in edge_w.items():
    if w >= MIN_EDGE_WEIGHT:
        G.add_edge(a, b, weight=w)
print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ---- 4. Detect "programme" communities ----------------------------------
# Louvain is near-linear and scales far better than greedy_modularity_communities.
communities = nxcom.louvain_communities(G, weight='weight', seed=42)
mod2com = {m: i for i, c in enumerate(communities) for m in c}
print(f"Detected {len(communities)} programme-like item clusters")

# Any item dropped during pruning (rare or hub) has no cluster assignment.
# Use student_modules_pruned (not the original) from here on so every item
# looked up in mod2com is guaranteed to exist.
student_modules = student_modules_pruned

# ---- 5. Score every student for cluster-switching -----------------------
rows = []
for pid, mods in student_modules.items():
    coms = [mod2com[m] for m in mods]
    cnt = Counter(coms)
    ranked = sorted(cnt.items(), key=lambda x: -x[1])
    primary_cluster, primary_count = ranked[0]
    secondary_cluster, secondary_count = (ranked[1] if len(ranked) > 1 else (None, 0))
    rows.append({
        'P_ID': pid,
        'n_items': len(mods),
        'n_clusters_touched': len(cnt),
        'primary_cluster': primary_cluster,
        'primary_count': primary_count,
        'secondary_cluster': secondary_cluster,
        'secondary_count': secondary_count,
    })

res = pd.DataFrame(rows)

# ---- 6. Flag the anomalies (strict: ratio + count) -----------------------
res['secondary_ratio'] = res['secondary_count'] / res['n_items']

# Strict thresholds: require a SUBSTANTIAL secondary cluster both in absolute
# terms (>=20 items) and as a share of total activity (>=30%). This favors
# precision over recall -> fewer false positives, only the clearest switches.
COUNT_THRESHOLD = 20
RATIO_THRESHOLD = 0.30

shape_shifters = res[
    (res['secondary_count'] >= COUNT_THRESHOLD) &
    (res['secondary_ratio'] >= RATIO_THRESHOLD)
].sort_values('secondary_count', ascending=False)

if pdim is not None:
    shape_shifters = shape_shifters.merge(pdim, on='P_ID', how='left')

print(f"Flagged {len(shape_shifters)} shape-shifter candidates "
      f"(strict: count>={COUNT_THRESHOLD}, ratio>={RATIO_THRESHOLD:.0%})")
shape_shifters.to_csv('shape_shifters.csv', index=False)
print(shape_shifters.head(20).to_string(index=False))

# P_ID-only export
shape_shifters[['P_ID']].to_csv('shape_shifters_pid_only.csv', index=False)

Using 'VIDEO_ID' as the module/content proxy column.
Keeping 19887 / 25761 items after pruning (dropped rare items < 5 students and hubs > 50% of students)
Graph built: 19887 nodes, 1757777 edges
Detected 115 programme-like item clusters
Flagged 65 shape-shifter candidates (strict: count>=20, ratio>=30%)
     P_ID  n_items  n_clusters_touched  primary_cluster  primary_count  secondary_cluster  secondary_count  secondary_ratio registration_date deregistration_date
P00012484      168                   2               77             88               30.0               80         0.476190        2023-09-26                 NaN
P00005821      238                   2              100            166               82.0               72         0.302521        2023-09-11                 NaN
P00009554      124                   2              100             63               82.0               61         0.491935        2023-09-04                 NaN
P00014094      157                   2        

In [67]:
shape_shifters[['P_ID']].to_csv('shape_shifters_pid_only.csv', index=False)